# Small Language Models (SLMs)

## What Are SLMs?
Small Language Models are LLMs under ~10B parameters, designed to run efficiently on consumer hardware, mobile devices, and edge environments.

## Why SLMs Matter

| Factor | SLMs | Large LLMs |
|--------|------|------------|
| Latency | <100ms (local) | 200-2000ms (API) |
| Cost | Near-zero after setup | $0.001-$0.06/1K tokens |
| Privacy | Fully local | Data leaves device |
| Offline | Yes | No |
| Quality | Task-specific excellent | General-purpose better |

## Model Landscape

| Model | Params | Creator | Key Strength |
|-------|--------|---------|-------------|
| Phi-4 | 14B | Microsoft | STEM, reasoning |
| Phi-3.5-mini | 3.8B | Microsoft | Tiny but powerful |
| Gemma 2 9B | 9B | Google | Quality/size ratio |
| Gemma 2 2B | 2B | Google | Edge deployment |
| Llama 3.2 3B | 3B | Meta | Strong baseline |
| Llama 3.2 1B | 1B | Meta | Mobile-ready |
| SmolLM2 1.7B | 1.7B | HuggingFace | Smallest useful |
| SmolLM2 360M | 360M | HuggingFace | On-device |
| Qwen2.5 3B | 3B | Alibaba | Multilingual |
| Qwen2.5 0.5B | 0.5B | Alibaba | Embedded |
| TinyLlama 1.1B | 1.1B | Tinyllama | Research |
| Mistral 7B | 7B | Mistral | Production-quality |

## Speculative Decoding

Speed up large models by using a small draft model to propose tokens, then verify with the target model:

1. Draft model generates $K$ tokens cheaply
2. Target model verifies all $K$ tokens in one forward pass
3. Accept tokens up to first mismatch

**Speedup**: 2-3x with minimal quality loss.

$$\text{speedup} \approx \frac{K+1}{1 + K(1-\alpha)}$$

where $\alpha$ is the acceptance rate.

In [1]:
# Running SLMs locally with Transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

def run_slm(model_id, prompt, max_new_tokens=200):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    pipe = pipeline('text-generation', model=model, tokenizer=tokenizer)
    result = pipe(prompt, max_new_tokens=max_new_tokens, temperature=0.7, do_sample=True)
    return result[0]['generated_text']

# Example with SmolLM2 (lightest useful model)
# run_slm("HuggingFaceTB/SmolLM2-1.7B-Instruct", "Explain backpropagation simply:")

print("SLM loading code ready. Models available:")
slm_models = [
    ("microsoft/phi-4", 14, "STEM tasks"),
    ("microsoft/Phi-3.5-mini-instruct", 3.8, "Tiny but capable"),
    ("google/gemma-2-9b-it", 9, "Quality/size ratio"),
    ("google/gemma-2-2b-it", 2, "Edge deployment"),
    ("meta-llama/Llama-3.2-3B-Instruct", 3, "Strong baseline"),
    ("meta-llama/Llama-3.2-1B-Instruct", 1, "Mobile-ready"),
    ("HuggingFaceTB/SmolLM2-1.7B-Instruct", 1.7, "Smallest useful"),
    ("Qwen/Qwen2.5-3B-Instruct", 3, "Multilingual"),
]
for model_id, size, desc in slm_models:
    print(f"  {size}B  {model_id.split('/')[1]:40s} {desc}")

SLM loading code ready. Models available:
  14B  phi-4                                    STEM tasks
  3.8B  Phi-3.5-mini-instruct                    Tiny but capable
  9B  gemma-2-9b-it                            Quality/size ratio
  2B  gemma-2-2b-it                            Edge deployment
  3B  Llama-3.2-3B-Instruct                    Strong baseline
  1B  Llama-3.2-1B-Instruct                    Mobile-ready
  1.7B  SmolLM2-1.7B-Instruct                    Smallest useful
  3B  Qwen2.5-3B-Instruct                      Multilingual


In [2]:
# Quantization with llama.cpp / GGUF format
# This is the most common way to run SLMs on CPU

GGUF_CODE = '''
# Install: pip install llama-cpp-python
# Download GGUF model from HuggingFace (e.g., TheBloke repos)

from llama_cpp import Llama

# Load quantized model (Q4_K_M = 4-bit, good quality/size trade-off)
llm = Llama(
    model_path="./llama-3.2-3b-instruct-q4_k_m.gguf",
    n_ctx=4096,       # context window
    n_threads=8,      # CPU threads
    n_gpu_layers=0,   # 0 = CPU only, -1 = all layers on GPU
    verbose=False
)

response = llm.create_chat_completion(
    messages=[{"role": "user", "content": "What is machine learning?"}],
    max_tokens=200,
    temperature=0.7
)
print(response["choices"][0]["message"]["content"])

# GGUF quantization levels:
# Q2_K  smallest, lowest quality
# Q4_K_M recommended, good balance
# Q5_K_M better quality
# Q8_0  near full precision, largest
# F16   full float16
'''
print(GGUF_CODE)


# Install: pip install llama-cpp-python
# Download GGUF model from HuggingFace (e.g., TheBloke repos)

from llama_cpp import Llama

# Load quantized model (Q4_K_M = 4-bit, good quality/size trade-off)
llm = Llama(
    model_path="./llama-3.2-3b-instruct-q4_k_m.gguf",
    n_ctx=4096,       # context window
    n_threads=8,      # CPU threads
    n_gpu_layers=0,   # 0 = CPU only, -1 = all layers on GPU
    verbose=False
)

response = llm.create_chat_completion(
    messages=[{"role": "user", "content": "What is machine learning?"}],
    max_tokens=200,
    temperature=0.7
)
print(response["choices"][0]["message"]["content"])

# GGUF quantization levels:
# Q2_K  smallest, lowest quality
# Q4_K_M recommended, good balance
# Q5_K_M better quality
# Q8_0  near full precision, largest
# F16   full float16



In [3]:
# ONNX Runtime for SLM inference
# Optimized for edge/mobile deployment

ONNX_CODE = '''
# pip install onnxruntime-genai
import onnxruntime_genai as og

# Phi-3.5 mini in ONNX format
model = og.Model("microsoft/Phi-3.5-mini-instruct-onnx")
tokenizer = og.Tokenizer(model)

prompt = "<|user|>\nExplain quantum computing<|end|>\n<|assistant|>\n"
input_tokens = tokenizer.encode(prompt)

params = og.GeneratorParams(model)
params.set_search_options(max_length=500, temperature=0.7)
params.input_ids = input_tokens

generator = og.Generator(model, params)
while not generator.is_done():
    generator.compute_logits()
    generator.generate_next_token()
    new_token = generator.get_next_tokens()[0]
    print(tokenizer.decode([new_token]), end='', flush=True)
'''
print(ONNX_CODE)


# pip install onnxruntime-genai
import onnxruntime_genai as og

# Phi-3.5 mini in ONNX format
model = og.Model("microsoft/Phi-3.5-mini-instruct-onnx")
tokenizer = og.Tokenizer(model)

prompt = "<|user|>
Explain quantum computing<|end|>
<|assistant|>
"
input_tokens = tokenizer.encode(prompt)

params = og.GeneratorParams(model)
params.set_search_options(max_length=500, temperature=0.7)
params.input_ids = input_tokens

generator = og.Generator(model, params)
while not generator.is_done():
    generator.compute_logits()
    generator.generate_next_token()
    new_token = generator.get_next_tokens()[0]
    print(tokenizer.decode([new_token]), end='', flush=True)



In [4]:
# MLX Apple Silicon optimized inference
MLX_CODE = '''
# pip install mlx mlx-lm
from mlx_lm import load, generate

# Load model (auto-converts to MLX format)
model, tokenizer = load("mlx-community/Llama-3.2-3B-Instruct-4bit")

response = generate(
    model, tokenizer,
    prompt="Explain neural networks in simple terms:",
    max_tokens=300,
    verbose=True
)
# Runs at full Metal GPU speed on M1/M2/M3/M4 Macs
# Llama 3.2 3B: ~100 tokens/sec on M2
'''
print(MLX_CODE)


# pip install mlx mlx-lm
from mlx_lm import load, generate

# Load model (auto-converts to MLX format)
model, tokenizer = load("mlx-community/Llama-3.2-3B-Instruct-4bit")

response = generate(
    model, tokenizer,
    prompt="Explain neural networks in simple terms:",
    max_tokens=300,
    verbose=True
)
# Runs at full Metal GPU speed on M1/M2/M3/M4 Macs
# Llama 3.2 3B: ~100 tokens/sec on M2



## Additional Learning Resources

### Papers
- [Phi-3 Technical Report](https://arxiv.org/abs/2404.14219)
- [Phi-4 Technical Report](https://arxiv.org/abs/2412.08905)
- [Gemma 2](https://arxiv.org/abs/2408.00118)
- [Speculative Decoding](https://arxiv.org/abs/2211.17192)
- [LLM.int8()](https://arxiv.org/abs/2208.07339) quantization paper

### Models
- [SmolLM2](https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B-Instruct)
- [Phi-4 on HF](https://huggingface.co/microsoft/phi-4)
- [Llama 3.2 1B](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct)

### Tools
- [Ollama](https://ollama.ai/) easiest local LLM runner
- [llama.cpp](https://github.com/ggerganov/llama.cpp)
- [LM Studio](https://lmstudio.ai/) GUI for local LLMs
- [MLX](https://github.com/ml-explore/mlx) Apple Silicon ML
- [ONNX Runtime GenAI](https://github.com/microsoft/onnxruntime-genai)